In [1]:
import coiled

import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import pytz
import dask
import re
import sparse
import time
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy

import pygwalker as pyg

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [3]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [ ]:
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=20,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r5.2xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="r5.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

In [ ]:
client.restart() 

In [4]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

/home/dagibbs22/miniforge3/envs/coiled_20250203/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41589 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:41589/status,
Dashboard: http://127.0.0.1:41589/status,Workers: 7
Total threads: 14,Total memory: 31.08 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41963,Workers: 7
Dashboard: http://127.0.0.1:41589/status,Total threads: 14
Started: Just now,Total memory: 31.08 GiB
Comm: tcp://127.0.0.1:40057,Total threads: 2
Dashboard: http://127.0.0.1:38371/status,Memory: 4.44 GiB
Nanny: tcp://127.0.0.1:42833,


In [ ]:
local_client.shutdown()

In [5]:
def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [6]:
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

In [7]:
# Node codes output from model. Covers entire decision tree. Make sure that node codes are right-padded with 0s to 7 digits! 
# Otherwise, only the node codes that are seven digits without 0s will be matched with the node code rasters and output. 
# TODO: I may have accidentally missed some node codes when copying them from the decision tree. Check!
node_codes = np.array([
    1110000, 1120000, 1210000, 1220000, 2111000, 2112000,
    2121100, 2121200, 2122100, 2122200, 2123100, 2123200,
    2124100, 2124200, 2125100, 2125200, 2211100, 2211200, 2212110, 2212120, 
    2212210, 2212220, 2213110, 2213120, 2213210, 2213220,
    2214100, 2214200, 2215100, 2215200, 2221100, 2221200, 2223100, 2223200,
    2222100, 2222200, 3110000, 3120000, 3211211, 3211212,
    3211221, 3211222, 3212111, 3212112, 3212121, 3212122,
    3212211, 3212212, 3212221, 3212222, 3221110, 3221120,
    3221210, 3221220, 3222111, 3222112, 3222121, 3222122,
    3222210, 3222220, 4100000, 4210000, 4220000, 4310000,
    4320000, 5100000, 5210000, 5220000, 5310000, 5320000],
dtype=np.uint32)

# # Node codes output from model for 2x2 test area in DRC (23_-5_25_-3)
# node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
#                        2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
#                        2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
#                       dtype=np.uint32)

In [8]:
# Extracts some metadata/chunk properties to add to the output dataframe
def parse_metadata_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__(\d+_-?\d+_\d+_-?\d+)__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_pixel_yr_(\d{4}_\d{4})\.tif$"
    match = re.search(pattern, uri)

    if match:
        chunk_id = match.group(1)
        variable = match.group(2)
        interval = match.group(3)
    else:
        interval, chunk_id, variable = None, None, None

    return interval, chunk_id, variable

In [21]:
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y': chunk_size}
    ).squeeze().persist()

    return xarray_chunks

In [10]:
def align_with_nodes(analysis_layer, nodes):
    analysis_layer_sub, nodes_aligned = xr.align(analysis_layer, nodes, join="inner")
    return analysis_layer_sub, nodes_aligned

In [11]:
# reduction using sum and count from https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6821fa2d-d658-800a-a371-aaa570dac95c
def xarray_reduction_sum_count(analysis_layer, node_data):

    reductions = {}

    for func in ["sum", "count"]:
        reduced = xarray_reduce(
            analysis_layer.band_data,
            node_data,
            func=func,
            keep_attrs=True,
            expected_groups=(node_codes),
            reindex=ReindexStrategy(
                blockwise=False,
                array_type=ReindexArrayType.SPARSE_COO
            ),
            fill_value=0
        )

        # Rename variables to reflect reduction type
        if isinstance(reduced, xr.Dataset):
            renamed = reduced.rename({var: f"{var}_{func}" for var in reduced.data_vars})
        else:  # it's a DataArray
            renamed = reduced.rename(f"{reduced.name}_{func}")

        reductions[func] = renamed

    # Merge results: handle Dataset or DataArray combinations
    result = xr.merge([r if isinstance(r, xr.Dataset) else r.to_dataset() for r in reductions.values()])

    return result

In [12]:
def xarray_reduction(analysis_layer, node_data):

    analysis_layer_by_node = xarray_reduce(
        analysis_layer.band_data,
        node_data,
        func='sum',
        keep_attrs=True,
        expected_groups=(node_codes),
        reindex=ReindexStrategy(
            blockwise=False, array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0   
    )

    return analysis_layer_by_node

In [48]:
# Creates output dataframe with separate columns for the sum and the pixel count
# Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6821fa2d-d658-800a-a371-aaa570dac95c
def create_output_year_df_sum_count(output_year_result, interval, output_pattern):
    """
    Converts a Dataset with sparse reduction outputs (e.g. sum and count) 
    into a tidy long-form DataFrame with one row per non-zero entry.
    """
    dfs = []
    for var in output_year_result.data_vars:
        arr = output_year_result[var]

        # Only works for 1D with state_node
        if not isinstance(arr.data, sparse.SparseArray):
            raise ValueError(f"Expected sparse array for variable '{var}'")

        coo = arr.data
        coords = coo.coords[0]
        values = coo.data
        state_nodes = arr.coords["state_node"].values[coords]

        df = pd.DataFrame({
            "state_node": state_nodes,
            var: values,
            "interval": interval,
            "output_pattern": output_pattern
        })

        dfs.append(df)

    # Merge on state_node, interval, output_pattern
    from functools import reduce
    df_merged = reduce(lambda left, right: pd.merge(left, right, on=["state_node", "interval", "output_pattern"]), dfs)

    return df_merged

In [13]:
def create_output_year_df(output_year_result, interval, ouput_pattern):

    output_year_result_sparse_data = output_year_result.data

    # Step 3: Extract coordinates and values
    dim_names = output_year_result.dims
    indices = output_year_result_sparse_data.coords
    output_year_values = output_year_result_sparse_data.data

    # Step 4: Map dimension indices to coordinate values
    output_year_coord_dict = {
        dim: output_year_result.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    output_year_coord_dict["value"] = output_year_values
    
    output_year_coord_dict = {
        dim: output_year_result.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    output_year_coord_dict["value"] = output_year_values
    
    output_year_df = pd.DataFrame(output_year_coord_dict)

    output_year_df["interval_end"] = interval
    output_year_df["output_pattern"] = ouput_pattern

    return output_year_df

In [14]:
# Reclassifies state nodes to broad categories
def classify_node(state_node):
    
    node_str = str(state_node)
    first_digit = int(node_str[0])
    # print(first_digit)

    # For broad classes that can be categorized using just the first digit
    one_digit_map = {
        1: 'forest_gain',
        2: 'forest_loss',
        4: 'cropland',
        5: 'grassland'
    }

    # For broad classes that need to be categorized using the first three digits
    three_digit_map = {
        321: 'disturbed_forest',
        322: 'stable_forest'
        # Add more as needed
    }
    
    if first_digit == 3:
        prefix = int(node_str[:3])
        # print(prefix)
        # print(two_digit_map.get(prefix, 'unknown_3x'))
        return three_digit_map.get(prefix, 'unknown_3x')
    else:
        return one_digit_map.get(first_digit, 'unknown')

Code to run zonal stats

In [65]:
# uri components

model_version = "version_0_3_2"
run_date = "20250507"
chunk_size = 2000
# model_version = "version_0_3_3"
# run_date = "20250511"
# chunk_size = 10000
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
# interval_end_years = [2016]
interval_end_years = [2016, 2017, 2018]
# interval_end_years = [2020]
# interval_end_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

# s3 folders for inputs
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/"

node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/"

analysis_layers_folders = [gross_emis_CO2_folder]
# analysis_layers_folders = [gross_emis_CO2_folder, gross_remv_all_pools_folder, net_flux_all_pools_CO2_folder]
# analysis_layers_folders = [gross_emis_CO2_folder, gross_emis_all_gases_folder, gross_remv_all_pools_folder, net_flux_all_pools_CO2_folder]

In [66]:
combined_df = pd.DataFrame()
analysis_start_time = time.time()

for focal_analysis_layer_folder in analysis_layers_folders:

    layer_start_time = time.time()

    for interval_end_year in interval_end_years:

        interval = f"{interval_end_year-1}_{interval_end_year}"
        # print(interval)

        # Creates a Pandas series of s3 uris for this specific analysis layer
        focal_analysis_layer_year_folder = focal_analysis_layer_folder.replace("INTERVAL", interval)
        focal_analysis_layer_year_uris = list_folder_uris(focal_analysis_layer_year_folder)

        # Creates a Pandas series of s3 uris for the relevant node codes
        node_folder = node_folder.replace("INTERVAL", interval)
        node_tile_year_uris = list_folder_uris(node_folder)

        # Gets input layer metadata, like the output pattern.
        # Note: chunk_id is for the first chunk being processed, not all chunks being processed.
        interval_from_inputs, chunk_id, output_pattern = parse_metadata_from_uri(focal_analysis_layer_year_uris)
        # print(output_pattern)
    
        print(f"Processing {(output_pattern)} for {interval} from {focal_analysis_layer_year_folder}: {timestr()}")
        layer_interval_start_time = time.time()

        print(f"   Reading {(output_pattern)} for {interval}: {timestr()}")
        layer_xarray_chunks = make_xarray_chunks(focal_analysis_layer_year_uris, chunk_size)
        node_xarray_chunks = make_xarray_chunks(node_tile_year_uris, chunk_size)
        # print("layer_xarray_chunks:", layer_xarray_chunks)
        # print("nodes_xarray_chunks:", nodes_xarray_chunks)

        print(f"   Aligning {(output_pattern)} for {interval}: {timestr()}")
        layer_sub, nodes_aligned = align_with_nodes(layer_xarray_chunks, node_xarray_chunks)
        # print("layer_sub:", layer_sub)
        # print("nodes_aligned:", nodes_aligned)
        
        node_data = nodes_aligned.band_data
        node_data.name = 'state_node'
        # print("node_data", node_data)

        print(f"   Reducing {(output_pattern)} for {interval}: {timestr()}")
        
        analysis_layer_by_node = xarray_reduction_sum_count(layer_sub, node_data)
        # analysis_layer_by_node = xarray_reduction(layer_sub, node_data)
        # print("analysis_layer_by_node:", analysis_layer_by_node)

        print(f"   Computing {(output_pattern)} for {interval}: {timestr()}")
        
        analysis_layer_result = analysis_layer_by_node.compute()
        # print(analysis_layer_result)

        print(f"   Creating and concatenating dataframe for {(output_pattern)} for {interval}: {timestr()}")
        
        output_year_df = create_output_year_df_sum_count(analysis_layer_result, interval_end_year, output_pattern)
        # print(output_year_df)
    
        combined_df = pd.concat([combined_df, output_year_df])

        layer_interval_end_time = time.time()
        print(f"   {(output_pattern)} for {interval} took {round(layer_interval_end_time - layer_interval_start_time)} seconds")

    layer_end_time = time.time()
    print(f"---{(output_pattern)} for all {len(interval_end_years)} intervals took {round(layer_end_time - layer_start_time)} seconds")
    print("")

combined_df = combined_df.reset_index(drop=True)
analysis_end_time = time.time()
print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")
combined_df['node_grp'] = combined_df['state_node'].apply(classify_node)
combined_df['state_node'] = 'n' + combined_df['state_node'].astype(str)
combined_df

Processing gross_emissions__all_C_pools__CO2_only__MgCO2 for 2015_2016 from s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_2/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/2015_2016/_pixel_yr/4000_pixels/20250507/: 20250513_14_41_24
   Reading gross_emissions__all_C_pools__CO2_only__MgCO2 for 2015_2016: 20250513_14_41_24
   Aligning gross_emissions__all_C_pools__CO2_only__MgCO2 for 2015_2016: 20250513_14_41_27
   Reducing gross_emissions__all_C_pools__CO2_only__MgCO2 for 2015_2016: 20250513_14_41_27
   Computing gross_emissions__all_C_pools__CO2_only__MgCO2 for 2015_2016: 20250513_14_41_27
   Creating and concatenating dataframe for gross_emissions__all_C_pools__CO2_only__MgCO2 for 2015_2016: 20250513_14_41_30
   gross_emissions__all_C_pools__CO2_only__MgCO2 for 2015_2016 took 6 seconds
Processing gross_emissions__all_C_pools__CO2_only__MgCO2 for 2016_2017 from s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_2/gross_

,state_node,band_data_sum,interval,output_pattern,band_data_count,node_grp
0,n2211100,1.060868e+04,2016,gross_emissions__all_C_pools__CO2_only__MgCO2,340,forest_loss
1,n2211200,2.899860e+04,2016,gross_emissions__all_C_pools__CO2_only__MgCO2,905,forest_loss
2,n2212110,9.996288e+06,2016,gross_emissions__all_C_pools__CO2_only__MgCO2,238927,forest_loss
3,n2212120,5.400350e+07,2016,gross_emissions__all_C_pools__CO2_only__MgCO2,1286841,forest_loss
4,n2212210,4.059757e+04,2016,gross_emissions__all_C_pools__CO2_only__MgCO2,2287,forest_loss
...,...,...,...,...,...,...
77,n5100000,1.068501e+03,2018,gross_emissions__all_C_pools__CO2_only__MgCO2,639,grassland
78,n5210000,2.831528e+01,2018,gross_emissions__all_C_pools__CO2_only__MgCO2,17,grassland
79,n5220000,1.727002e+03,2018,gross_emissions__all_C_pools__CO2_only__MgCO2,1026,grassland
80,n5310000,1.780022e+04,2018,gross_emissions__all_C_pools__CO2_only__MgCO2,9931,grassland


In [67]:
combined_df[(combined_df.output_pattern == 'gross_emissions__all_C_pools__CO2_only__MgCO2') 
& (combined_df.state_node == 'n3222112')]

,state_node,band_data_sum,interval,output_pattern,band_data_count,node_grp
37,n3222112,43371928.0,2017,gross_emissions__all_C_pools__CO2_only__MgCO2,1476768,stable_forest
67,n3222112,60203100.0,2018,gross_emissions__all_C_pools__CO2_only__MgCO2,1971748,stable_forest


In [63]:
walker = pyg.walk(combined_df)

Box(children=(HTML(value='<div id="ifr-pyg-1" style="height: auto">\n    <head>\n        <meta http-equiv="Con…